In [ ]:
import numpy as np
import torch
import torch.nn as nn


RAND_SEED = 5904
torch.manual_seed(RAND_SEED)
torch.cuda.manual_seed(RAND_SEED)

In [ ]:
split_data = np.load("../data/train_test_splits.npz", allow_pickle=True)

train_X = split_data["train_X"]
test_X = split_data["test_X"]

train_Y = split_data["train_Y"]
test_Y = split_data["test_Y"]

train_probs_y = train_Y[:,0]
train_times_t = train_Y[:,1]

test_probs_y = test_Y[:,0]
test_times_t = test_Y[:,1]

NUM_FEATURES = train_X.shape[1]

In [ ]:
def get_features_over_time(feature_data_X, num_time_steps):
    num_rows = feature_data_X.shape[0]
    num_features = feature_data_X.shape[1]

    # Fill of 0 values for missing inputs at times
    nan_array = np.full((num_rows, num_time_steps-1, num_features), 0.0)

    # Add time dimension to existing feature vectors
    single_features = np.expand_dims(feature_data_X, axis=1)

    # Concatenation along time dimension
    ret = np.concat((single_features, nan_array), axis=1)

    return ret

def get_outcomes_over_time(outcome_data, time_data, num_time_steps):

    # Apply floor function to times to get discrete integers
    floor_times = np.floor(time_data).astype(int)

    # Creating a sequence for each case
    outcomes_over_time = np.zeros((outcome_data.shape[0], num_time_steps))

    # Determine the indices where the outcome is 1.0
    positive_cases = np.where(outcome_data==1.0)[0]

    # 1.0 at event, like impulse function
    # outcomes_over_time[positive_cases, floor_times[positive_cases]] = 1.0

    # 1.0 at event and after, like step function
    time_steps = np.arange(num_time_steps)
    event_steps = (time_steps >= floor_times[positive_cases][:, None]).astype(float)
    outcomes_over_time[positive_cases] = event_steps

    # Add column dimension
    outcomes_over_time = np.expand_dims(outcomes_over_time, -1)

    return outcomes_over_time


In [ ]:
num_time_steps = 62

train_features_X = torch.tensor(get_features_over_time(train_X, num_time_steps), dtype=torch.float32)
test_features_X = torch.tensor(get_features_over_time(test_X, num_time_steps), dtype=torch.float32)

train_outcomes_Y = torch.tensor(get_outcomes_over_time(train_probs_y, train_times_t, num_time_steps), dtype=torch.float32)
test_outcomes_Y = torch.tensor(get_outcomes_over_time(test_probs_y, test_times_t, num_time_steps), dtype=torch.float32)

train_data = torch.utils.data.TensorDataset(train_features_X, train_outcomes_Y)
test_data = torch.utils.data.TensorDataset(test_features_X, test_outcomes_Y)

trainloader = torch.utils.data.DataLoader(train_data, batch_size=16, shuffle=True)
testloader = torch.utils.data.DataLoader(test_data, batch_size=16, shuffle=False)

In [ ]:
class ProbabilityPredictorGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(ProbabilityPredictorGRU, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size

        self.gru = nn.GRU(input_size, hidden_size, num_layers)
        self.fc1 = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, features_over_time):
        x, _ = self.gru(features_over_time)
        x = self.fc1(x)
        x = self.sigmoid(x)

        return x


In [ ]:
# gru_model = ProbabilityPredictorGRU(NUM_FEATURES, 16, 2)
# gru_model(test_features_X).shape

In [ ]:
BATCH_SIZE = 16
BATCH_PRINT_STEP = 40
NUM_EPOCHS = 200


if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Training Over", len(train_X), "follow ups")

model = ProbabilityPredictorGRU(NUM_FEATURES, 16, 2)

model = model.to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

for epoch in range(0, NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        features, targets = data

        features = features.to(device)
        targets = targets.to(device)
        
        # Zero gradients
        optimizer.zero_grad()

        # Forward
        output = model(features)

        # Backward
        loss = criterion(output, targets)
        loss.backward()
        
        # Optimize
        optimizer.step()

        running_loss += loss.item()

        if i % BATCH_PRINT_STEP == (BATCH_PRINT_STEP - 1):
            curr_loss = running_loss / BATCH_PRINT_STEP
            epochBatchLossPrint = "Epoch: {} Batch: {} Loss: {:.5f}".format(epoch + 1, i + 1, curr_loss)
            print(epochBatchLossPrint)
            running_loss = 0.0